# 01 — Chunking + ChromaDB Indexing

Reads scraped ToS/Privacy documents, chunks them using LangChain's RecursiveCharacterTextSplitter,
embeds with OpenAI, and stores in ChromaDB for retrieval.


## Step 1: Install Dependencies

In [1]:
!pip install chromadb openai tiktoken pandas langchain langchain-text-splitters -q
print("✓ Dependencies installed")


✓ Dependencies installed


## Step 2: Imports + Config

In [ ]:
import os
import json
import re
from pathlib import Path

import chromadb
from chromadb.utils import embedding_functions
from langchain_text_splitters import RecursiveCharacterTextSplitter

CORPUS_DIR  = "tos_corpus"
CHROMA_DIR  = "chroma_db"
MANIFEST    = os.path.join(CORPUS_DIR, "manifest.json")

CHUNK_SIZE    = 2048  
CHUNK_OVERLAP = 256   

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "YOUR_API_KEY_HERE")
EMBED_MODEL    = "text-embedding-3-small"


splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ". ", " ", ""]
)

print("✓ Config ready")
print(f"  Chunk size:    {CHUNK_SIZE} chars")
print(f"  Chunk overlap: {CHUNK_OVERLAP} chars")
print(f"  Embed model:   {EMBED_MODEL}")


/Users/ghaithalramahi/Downloads/DS593 Final Project/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✓ Config ready
  Chunk size:    2048 chars
  Chunk overlap: 256 chars
  Embed model:   text-embedding-3-small


## Step 3: Load Manifest

In [3]:
with open(MANIFEST) as f:
    manifest = json.load(f)

print(f"✓ Loaded {len(manifest)} documents\n")
for doc in manifest:
    print(f"  {doc['company']:15s} | {doc['doc_type']:8s} | {doc['char_count']:,} chars")


✓ Loaded 25 documents

  spotify         | tos      | 54,254 chars
  spotify         | privacy  | 39,102 chars
  tiktok          | tos      | 41,558 chars
  tiktok          | privacy  | 27,650 chars
  airbnb          | tos      | 255,682 chars
  airbnb          | privacy  | 1,632 chars
  netflix         | tos      | 56,237 chars
  netflix         | privacy  | 69,870 chars
  twitter_x       | tos      | 58,844 chars
  twitter_x       | privacy  | 33,347 chars
  google          | tos      | 28,244 chars
  google          | privacy  | 64,193 chars
  paypal          | tos      | 158,439 chars
  paypal          | privacy  | 117,212 chars
  discord         | tos      | 59,562 chars
  discord         | privacy  | 35,521 chars
  snapchat        | tos      | 98,842 chars
  snapchat        | privacy  | 32,875 chars
  youtube         | tos      | 23,990 chars
  youtube         | privacy  | 64,193 chars
  robinhood       | tos      | 3,405 chars
  tinder          | tos      | 100,317 chars
  tinde

## Step 4: Preview Chunking on One Document


In [4]:
# Preview on Spotify ToS
with open(manifest[0]["filepath"], encoding="utf-8") as f:
    text = f.read()

chunks = splitter.split_text(text)

print(f"Document:       {manifest[0]['filepath']}")
print(f"Total chars:    {len(text):,}")
print(f"Total chunks:   {len(chunks)}")
print(f"Avg chunk size: {sum(len(c) for c in chunks) // len(chunks):,} chars")
print()

for i, chunk in enumerate(chunks[:3]):
    print(f"── Chunk {i} ({len(chunk):,} chars) ──")
    print(chunk[:400])
    print("...")
    print()


Document:       tos_corpus/spotify_tos_2026-04-17.txt
Total chars:    54,254
Total chunks:   32
Avg chunk size: 1,774 chars

── Chunk 0 (1,934 chars) ──
Terms and Conditions of Use - Spotify
Spotify Terms of Use
Last Updated: August 26, 2025
Introduction
The Spotify Service
Your Use of the Spotify Service
Content and Intellectual Property Rights
Customer Support, Information, Questions, and Complaints
Problems and Disputes
About These Terms
1. Introduction
Please read these Terms of Use ("
Terms
") carefully as they govern your use of (which inclu
...

── Chunk 1 (2,024 chars) ──
Service provider
These Terms are between you and
Spotify USA Inc.
, 4 World Trade Center, 150 Greenwich Street, 62nd Floor, New York, NY, 10007.
Age and eligibility requirements
BY USING THE SPOTIFY SERVICE, YOU AFFIRM THAT YOU ARE 18 YEARS OR OLDER TO ENTER INTO THESE TERMS, OR, IF YOU ARE NOT, THAT YOU ARE 13 YEARS OR OLDER AND HAVE OBTAINED PARENTAL OR GUARDIAN CONSENT TO ENTER INTO THESE TERMS
...

── Chun

## Step 5: Metadata Helper

In [5]:
def extract_metadata_from_filename(filename: str) -> dict:
    """
    Parse company, doc_type, date from filename.
    Format: {company}_{doc_type}_{date}.txt
    e.g. spotify_privacy_2026-04-17.txt
    """
    stem  = Path(filename).stem
    parts = stem.split('_')
    date     = parts[-1]
    doc_type = parts[-2]
    company  = '_'.join(parts[:-2])
    return {"company": company, "doc_type": doc_type, "scrape_date": date}

print("✓ Metadata helper ready")


✓ Metadata helper ready


## Step 6: Set Up ChromaDB

In [6]:
openai_ef = embedding_functions.OpenAIEmbeddingFunction(
    api_key=OPENAI_API_KEY,
    model_name=EMBED_MODEL,
)

client = chromadb.PersistentClient(path=CHROMA_DIR)

# Delete existing collection if re-running to avoid duplicates
try:
    client.delete_collection("tos_documents")
    print("✓ Deleted existing collection (fresh start)")
except:
    pass

collection = client.get_or_create_collection(
    name="tos_documents",
    embedding_function=openai_ef,
    metadata={"hnsw:space": "cosine"},
)

print(f"✓ ChromaDB collection ready: '{collection.name}'")
print(f"  Saved to: {CHROMA_DIR}/")


✓ Deleted existing collection (fresh start)
✓ ChromaDB collection ready: 'tos_documents'
  Saved to: chroma_db/


## Step 7: Index All Documents

In [7]:
total_chunks = 0
skipped      = 0

for doc in manifest:
    filepath = doc["filepath"]
    filename = Path(filepath).name

    if not os.path.exists(filepath):
        print(f"✗ File not found: {filepath}, skipping")
        skipped += 1
        continue

    meta     = extract_metadata_from_filename(filename)
    company  = meta["company"]
    doc_type = meta["doc_type"]
    date     = meta["scrape_date"]

    print(f"\n── {company.upper()} ({doc_type}) ──")

    with open(filepath, encoding="utf-8") as f:
        text = f.read()

    chunks = splitter.split_text(text)
    print(f"  {len(text):,} chars → {len(chunks)} chunks")

    ids       = []
    documents = []
    metadatas = []

    for i, chunk in enumerate(chunks):
        chunk_id = f"{company}_{doc_type}_{date}_{i:04d}"
        ids.append(chunk_id)
        documents.append(chunk)
        metadatas.append({
            "company":     company,
            "doc_type":    doc_type,
            "scrape_date": date,
            "chunk_index": i,
            "source":      filename,
        })

    # Upsert in batches of 100
    batch_size = 100
    for batch_start in range(0, len(ids), batch_size):
        batch_end = batch_start + batch_size
        collection.upsert(
            ids=ids[batch_start:batch_end],
            documents=documents[batch_start:batch_end],
            metadatas=metadatas[batch_start:batch_end],
        )

    print(f"  ✓ Indexed {len(chunks)} chunks")
    total_chunks += len(chunks)

print(f"\n{'='*50}")
print(f"✓ Indexing complete")
print(f"  Total chunks indexed: {total_chunks}")
print(f"  Documents skipped:    {skipped}")
print(f"  Total in collection:  {collection.count()}")



── SPOTIFY (tos) ──
  54,254 chars → 32 chunks
  ✓ Indexed 32 chunks

── SPOTIFY (privacy) ──
  39,102 chars → 22 chunks
  ✓ Indexed 22 chunks

── TIKTOK (tos) ──
  41,558 chars → 27 chunks
  ✓ Indexed 27 chunks

── TIKTOK (privacy) ──
  27,650 chars → 16 chunks
  ✓ Indexed 16 chunks

── AIRBNB (tos) ──
  255,682 chars → 151 chunks
  ✓ Indexed 151 chunks

── AIRBNB (privacy) ──
  1,632 chars → 1 chunks
  ✓ Indexed 1 chunks

── NETFLIX (tos) ──
  56,237 chars → 36 chunks
  ✓ Indexed 36 chunks

── NETFLIX (privacy) ──
  69,870 chars → 40 chunks
  ✓ Indexed 40 chunks

── TWITTER_X (tos) ──
  58,844 chars → 37 chunks
  ✓ Indexed 37 chunks

── TWITTER_X (privacy) ──
  33,347 chars → 19 chunks
  ✓ Indexed 19 chunks

── GOOGLE (tos) ──
  28,244 chars → 16 chunks
  ✓ Indexed 16 chunks

── GOOGLE (privacy) ──
  64,193 chars → 37 chunks
  ✓ Indexed 37 chunks

── PAYPAL (tos) ──
  158,439 chars → 92 chunks
  ✓ Indexed 92 chunks

── PAYPAL (privacy) ──
  117,212 chars → 67 chunks
  ✓ Indexed 67 c

## Step 8: Inspect Collection

In [8]:
import pandas as pd

all_meta = collection.get()["metadatas"]
df       = pd.DataFrame(all_meta)

summary = df.groupby(["company", "doc_type"]).size().reset_index(name="chunks")
print(f"Collection summary ({collection.count()} total chunks):\n")
print(summary.to_string(index=False))


Collection summary (903 total chunks):

  company doc_type  chunks
   airbnb  privacy       1
   airbnb      tos     151
  discord  privacy      20
  discord      tos      36
   google  privacy      37
   google      tos      16
 linkedin  privacy      24
 linkedin      tos      19
  netflix  privacy      40
  netflix      tos      36
   paypal  privacy      67
   paypal      tos      92
robinhood      tos       2
 snapchat  privacy      19
 snapchat      tos      58
  spotify  privacy      22
  spotify      tos      32
   tiktok  privacy      16
   tiktok      tos      27
   tinder  privacy      20
   tinder      tos      61
twitter_x  privacy      19
twitter_x      tos      37
  youtube  privacy      37
  youtube      tos      14


## Step 9: Sanity Check Test Retrieval



In [9]:
def query_collection(question: str, company: str, doc_type: str = None, n_results: int = 3):
    if doc_type:
        where = {
            "$and": [
                {"company": company},
                {"doc_type": doc_type},
            ]
        }
    else:
        where = {"company": company}

    results = collection.query(
        query_texts=[question],
        n_results=n_results,
        where=where,
    )

    print(f"Query:  '{question}'")
    print(f"Filter: company={company}, doc_type={doc_type or 'both'}\n")

    docs = results.get("documents", [[]])[0]
    metas = results.get("metadatas", [[]])[0]
    distances = results.get("distances", [[]])[0]

    if not docs:
        print("No matching results found.\n")
        return

    for i, (doc, meta, distance) in enumerate(zip(docs, metas, distances), start=1):
        print(f"── Result {i} (distance: {distance:.4f}) ──")
        print(f"   Source: {meta['source']} | chunk {meta['chunk_index']}")
        print(f"   {doc[:300]}...")
        print()

# Test 1
query_collection(
    question="Can Spotify sell my personal data to third parties?",
    company="spotify",
    doc_type="privacy",
)

Query:  'Can Spotify sell my personal data to third parties?'
Filter: company=spotify, doc_type=privacy

── Result 1 (distance: 0.3472) ──
   Source: spotify_privacy_2026-04-17.txt | chunk 14
   You or another user can share certain information on third party services, like social media or messaging platforms. This includes:
your profile
any content you post on Spotify and details about that content
your playlists and any associated titles, descriptions and images
When this sharing occurs, ...

── Result 2 (distance: 0.3530) ──
   Source: spotify_privacy_2026-04-17.txt | chunk 15
   Other Spotify users
User Data
Usage Data
Message Data
To share information about your use of the Spotify Service with other Spotify users. These could include your followers on Spotify.
For example, under ‘Social’ settings you can choose to share your recently played artists and your playlists on yo...

── Result 3 (distance: 0.3626) ──
   Source: spotify_privacy_2026-04-17.txt | chunk 3
   Spotify needs th

In [10]:
# Test 2
query_collection(
    question="Can Discord delete my account without warning?",
    company="discord",
    doc_type="tos",
)


Query:  'Can Discord delete my account without warning?'
Filter: company=discord, doc_type=tos

── Result 1 (distance: 0.3859) ──
   Source: discord_tos_2026-04-17.txt | chunk 11
   Don’t use the services to do harm to Discord.
Among other things, this includes trying to gain access to, intentionally overburdening, or attacking our systems; scraping our services without our written consent, including by using any robot, spider, crawler, scraper, or other automatic device, proce...

── Result 2 (distance: 0.3988) ──
   Source: discord_tos_2026-04-17.txt | chunk 4
   Discord also takes user safety seriously. We provide features and services that are designed to help keep users safe.  Certain features and settings may have different defaults depending on your age or where you live, and some may not be optional at all. We also provide tools and resources like
Fami...

── Result 3 (distance: 0.4136) ──
   Source: discord_tos_2026-04-17.txt | chunk 12
   Disabling your account limits the pro

In [11]:
# Test 3
query_collection(
    question="Does TikTok own the videos I upload?",
    company="tiktok",
)


Query:  'Does TikTok own the videos I upload?'
Filter: company=tiktok, doc_type=both

── Result 1 (distance: 0.3738) ──
   Source: tiktok_tos_2026-04-17.txt | chunk 7
   Due to the nature of generative AI, output may not be unique to a specific user, and your ownership of Output does not extend to other users' output.
If you choose to submit comments, ideas, or feedback to us, you give us permission to use them in connection with the Platform and any other services ...

── Result 2 (distance: 0.3817) ──
   Source: tiktok_tos_2026-04-17.txt | chunk 6
   engage in inauthentic commercial behaviors, such as by operating spam or impersonation accounts or any other means further detailed in our
Community Guidelines
,
submit appeals, reports, notices or complaints which are unfounded,
scrape, crawl, export or otherwise extract any data or content in any ...

── Result 3 (distance: 0.4173) ──
   Source: tiktok_tos_2026-04-17.txt | chunk 9
   We retain all our respective rights, title, and inte

## Step 10: Build Additional Chunking Collections

Build two more ChromaDB collections using different chunking strategies.


- `chroma_db_small`    — 512 chars, 64 overlap (fine-grained)
- `chroma_db_sentence` — NLTK sentence boundaries (clause-preserving)



In [12]:
import nltk
from pathlib import Path
from langchain_text_splitters import RecursiveCharacterTextSplitter, NLTKTextSplitter

nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

small_splitter = RecursiveCharacterTextSplitter(
    chunk_size=512,
    chunk_overlap=64,
    separators=["\n\n", "\n", ". ", " ", ""],
)

sentence_splitter = NLTKTextSplitter(
    separator=" ",
    chunk_size=2048,
    chunk_overlap=256,
)

EXTRA_STRATEGIES = {
    "chroma_db_small": small_splitter,
    "chroma_db_sentence": sentence_splitter,
}

print("✓ Additional splitters ready")
print("  small:    512 chars, 64 overlap")
print("  sentence: NLTK sentence boundaries, ~2048 chars, 256 overlap")


✓ Additional splitters ready
  small:    512 chars, 64 overlap
  sentence: NLTK sentence boundaries, ~2048 chars, 256 overlap


In [13]:
def build_extra_collection(chroma_dir, splitter, manifest, openai_ef):
    import chromadb

    client = chromadb.PersistentClient(path=chroma_dir)

    try:
        client.delete_collection("tos_documents")
        print(f"  Deleted existing collection in {chroma_dir}")
    except Exception:
        pass

    col = client.get_or_create_collection(
        name="tos_documents",
        embedding_function=openai_ef,
        metadata={"hnsw:space": "cosine"},
    )

    total = 0

    for doc in manifest:
        filepath = doc["filepath"]
        if not os.path.exists(filepath):
            print(f"  Skipping missing file: {filepath}")
            continue

        filename = Path(filepath).name
        meta = extract_metadata_from_filename(filename)

        with open(filepath, encoding="utf-8") as f:
            text = f.read()

        chunks = [chunk.strip() for chunk in splitter.split_text(text) if chunk.strip()]

        ids = []
        documents = []
        metadatas = []

        for i, chunk in enumerate(chunks):
            ids.append(f"{meta['company']}_{meta['doc_type']}_{meta['scrape_date']}_{i:04d}")
            documents.append(chunk)
            metadatas.append({
                "company": meta["company"],
                "doc_type": meta["doc_type"],
                "scrape_date": meta["scrape_date"],
                "chunk_index": i,
                "source": filename,
            })

        for start in range(0, len(ids), 100):
            col.upsert(
                ids=ids[start:start + 100],
                documents=documents[start:start + 100],
                metadatas=metadatas[start:start + 100],
            )

        total += len(chunks)
        print(f"  {filename}: {len(chunks)} chunks")

    print(f"  ✓ {chroma_dir}: {total} chunks indexed ({col.count()} in collection)")
    return col


for chroma_dir, splitter in EXTRA_STRATEGIES.items():
    print(f"\nBuilding {chroma_dir}...")
    build_extra_collection(chroma_dir, splitter, manifest, openai_ef)

print("\n✓ All additional collections built")
print("  Ready for chunking comparison in 03_evaluation.ipynb")



Building chroma_db_small...
  Deleted existing collection in chroma_db_small
  spotify_tos_2026-04-17.txt: 167 chunks
  spotify_privacy_2026-04-17.txt: 94 chunks
  tiktok_tos_2026-04-17.txt: 118 chunks
  tiktok_privacy_2026-04-17.txt: 76 chunks
  airbnb_tos_2026-04-17.txt: 772 chunks
  airbnb_privacy_2026-04-17.txt: 4 chunks
  netflix_tos_2026-04-17.txt: 177 chunks
  netflix_privacy_2026-04-17.txt: 201 chunks
  twitter_x_tos_2026-04-17.txt: 178 chunks
  twitter_x_privacy_2026-04-17.txt: 91 chunks
  google_tos_2026-04-17.txt: 67 chunks
  google_privacy_2026-04-17.txt: 165 chunks
  paypal_tos_2026-04-17.txt: 452 chunks
  paypal_privacy_2026-04-17.txt: 323 chunks
  discord_tos_2026-04-17.txt: 179 chunks
  discord_privacy_2026-04-17.txt: 107 chunks
  snapchat_tos_2026-04-17.txt: 287 chunks
  snapchat_privacy_2026-04-17.txt: 88 chunks
  youtube_tos_2026-04-17.txt: 65 chunks
  youtube_privacy_2026-04-17.txt: 165 chunks
  robinhood_tos_2026-04-17.txt: 8 chunks
  tinder_tos_2026-04-17.txt: 29

Created a chunk of size 4186, which is longer than the specified 2048


  google_privacy_2026-04-17.txt: 36 chunks


Created a chunk of size 2862, which is longer than the specified 2048
Created a chunk of size 2383, which is longer than the specified 2048


  paypal_tos_2026-04-17.txt: 88 chunks
  paypal_privacy_2026-04-17.txt: 66 chunks
  discord_tos_2026-04-17.txt: 34 chunks


...

  robinhood_tos_2026-04-17.txt: 1 chunks


Created a chunk of size 2090, which is longer than the specified 2048
Created a chunk of size 2156, which is longer than the specified 2048


  tinder_tos_2026-04-17.txt: 58 chunks


Created a chunk of size 2294, which is longer than the specified 2048


  tinder_privacy_2026-04-17.txt: 20 chunks
  linkedin_tos_2026-04-17.txt: 19 chunks
  linkedin_privacy_2026-04-17.txt: 24 chunks
  ✓ chroma_db_sentence: 872 chunks indexed (872 in collection)

✓ All additional collections built
  Ready for chunking comparison in 03_evaluation.ipynb
